In [4]:
# STAGE 1 (FIXED): DATA PREPARATION & SMART PATH FINDING
# ------------------------------------------------------
import os
import zipfile
import pandas as pd
import numpy as np
from PIL import Image
from google.colab import drive
import shutil

# 1. Mount Drive (Force remount to ensure connection)
drive.mount('/content/drive', force_remount=True)

# CONFIG
PROJECT_PATH = '/content/drive/My Drive/Final Year Project'
ZIP_PATH = os.path.join(PROJECT_PATH, 'dataset.zip')
EXTRACT_ROOT = '/content/dataset_extraction' # Temp folder for extraction
OUTPUT_PATH = os.path.join(PROJECT_PATH, 'Processed_Data')

os.makedirs(OUTPUT_PATH, exist_ok=True)

# 2. Extract Dataset
if not os.path.exists(EXTRACT_ROOT):
    print(f"Extracting zip to {EXTRACT_ROOT}...")
    try:
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(EXTRACT_ROOT)
        print("✅ Extraction done.")
    except FileNotFoundError:
        print(f"❌ ERROR: Zip file not found at {ZIP_PATH}")
        print("Please check if 'NutriAI_Dataset.zip' is inside 'NutriAI_Project' folder in Drive.")
        # Stop execution here if zip is missing
        raise

# 3. SMART PATH FINDER (The Fix)
# This loop looks for the folder that actually contains 'train', 'val', or 'test'
dataset_root = None
for root, dirs, files in os.walk(EXTRACT_ROOT):
    if 'train' in dirs and 'val' in dirs:
        dataset_root = root
        print(f"✅ Found dataset root at: {dataset_root}")
        break

if dataset_root is None:
    print("❌ ERROR: Could not find 'train' and 'val' folders inside the zip.")
    print("Current folder structure:")
    print(os.listdir(EXTRACT_ROOT))
    raise FileNotFoundError("Dataset structure is invalid.")

# 4. Clean & Index Data
data = []
print("Scanning and cleaning images...")

valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.PNG'}

# Loop through train, val, test
for split in ['train', 'val', 'test']:
    split_path = os.path.join(dataset_root, split)
    if not os.path.exists(split_path):
        print(f"⚠️ Warning: {split} folder not found.")
        continue

    # Loop through Disease Classes (e.g., Tomato_healthy)
    for class_name in os.listdir(split_path):
        class_path = os.path.join(split_path, class_name)
        if not os.path.isdir(class_path): continue

        # Loop through Images
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            # Check extension
            ext = os.path.splitext(img_name)[1]
            if ext not in valid_extensions: continue

            try:
                # Verify Image works
                with Image.open(img_path) as img:
                    img.verify()

                # Simulate Environmental Data
                # Logic: Bacterial/Fungal = High Humidity
                is_humid = 1 if 'bacterial' in class_name.lower() or 'blight' in class_name.lower() else 0
                humidity = np.random.uniform(70, 95) if is_humid else np.random.uniform(40, 60)
                temp = np.random.uniform(20, 35)
                ph = np.random.uniform(5.5, 7.0)

                data.append({
                    'filepath': img_path,
                    'label': class_name,
                    'split': split,
                    'humidity': humidity,
                    'temperature': temp,
                    'ph': ph
                })
            except Exception as e:
                # If image is corrupt, ignore it
                pass

# 5. Create DataFrame & Save
df = pd.DataFrame(data)

if len(df) == 0:
    print("❌ ERROR: No images found even after fixing path!")
    print("Check: Are the folders inside 'train' empty?")
else:
    print(f"✅ Success! Total valid images: {len(df)}")
    print(f"Classes found: {len(df['label'].unique())}")
    print(df['label'].unique())

    # Save metadata to Drive for next stages
    df.to_csv(os.path.join(OUTPUT_PATH, 'full_dataset_metadata.csv'), index=False)
    print("Stage 1 Complete. Metadata saved to Drive.")

Mounted at /content/drive
Extracting zip to /content/dataset_extraction...
✅ Extraction done.
✅ Found dataset root at: /content/dataset_extraction/dataset
Scanning and cleaning images...
✅ Success! Total valid images: 22998
Classes found: 28
['Tomato septoria leaf spot' 'Tomato healthy' 'Chilli Healthy Leaf'
 'Potato___Late_blight' 'Cotton curl_virus' 'Maize fall armyworm'
 'Potato___healthy' 'Tomato verticulium wilt' 'Chilli Curl Virus'
 'Tomato leaf curl' 'Cotton bacterial_blight' 'Chilli Bacterial Spot'
 'Tomato leaf blight' 'Cotton fussarium_wilt' 'Maize leaf blight'
 'Maize grasshoper' 'Chilli Nutrition Deficiency' 'Cotton healthy'
 'Potato___Early_blight' 'Chilli Cercospora Leaf Spot'
 'Maize streak virus' 'Maize healthy' 'Chilli White spot'
 'Maize leaf beetle' 'Maize leaf spot' 'Potato_healthy'
 'Potato_Late_blight' 'Potato_Early_blight']
Stage 1 Complete. Metadata saved to Drive.
